# 1. Compréhension métier et des données

Le développement d'applications de machine learning débute par la délimitation du périmètre du projet, la définition des critères de succès et une vérification de la qualité des données. Cette première phase a pour objectif d'évaluer la faisabilité du projet.

Dans le contexte de la guillotine fiscale, une estimation précise de la valeur des biens immobiliers est primordiale : un bien mal évalué peut faire basculer un contribuable d'un côté ou de l'autre d'un seuil fiscal, avec des conséquences importantes sur sa charge d'imposition.


## 1.1 Chargement des données

Nous chargeons séparément le jeu d'entraînement (`train.csv`) et le jeu de validation (`test.csv`), en utilisant la colonne `Id` comme index. Cette séparation est fondamentale : le jeu d'entraînement contient la variable cible (`SalePrice`) qui nous permet d'apprendre les relations entre les caractéristiques d'un bien et son prix, tandis que le jeu de validation représente les données "inconnues" sur lesquelles nous devrons faire nos prédictions finales (soumission Kaggle).

Le jeu d'entraînement compte **1460 observations** et **80 variables**, ce qui constitue une base solide pour entraîner un modèle de régression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.gofplots import qqplot

In [ ]:
train_raw = pd.read_csv('Data/train.csv', index_col = 'Id')
validation_raw = pd.read_csv('Data/test.csv', index_col = 'Id')

## 1.2 Analyse exploratoire des données (EDA)

### Aperçu du jeu de données

Un premier aperçu avec `.head()` révèle la nature hétérogène des données : on y trouve des variables **numériques continues** (surface habitable, superficie du terrain), des variables **catégorielles** (type de zonage, matériaux de toiture) et des variables **temporelles** (année de construction, année de vente). Cette diversité imposera un traitement différencié selon le type de chaque variable.

On remarque également dès cette étape la présence de valeurs manquantes (`NaN`), notamment dans la colonne `Alley`, ce qui confirme la nécessité d'une stratégie d'imputation.

In [ ]:
train_raw.head()

### Analyse de la variable cible : `SalePrice`

Les statistiques descriptives révèlent plusieurs informations clés :
- **Moyenne (180 921 $) > Médiane (163 000 $)** : cet écart indique une asymétrie positive (skewness), typique des distributions de prix immobiliers, où quelques biens très chers tirent la moyenne vers le haut.
- **Écart-type de 79 442 $** : la variabilité est élevée, ce qui laisse supposer que plusieurs facteurs (localisation, surface, qualité) jouent un rôle important dans la détermination du prix.
- **Min à 34 900 $ et max à 755 000 $** : l'amplitude est très large, ce qui confirme la présence potentielle de valeurs extrêmes.

Cette asymétrie est problématique pour une régression linéaire, qui suppose des résidus normalement distribués. Nous devrons donc transformer la variable cible.

In [ ]:
# descriptive statistics summary
train_raw['SalePrice'].describe()

Visualisons la distribution des prix de vente. Une asymétrie visible à l'oeil nu justifiera de ne pas utiliser les prix bruts directement comme variable cible.

In [ ]:
# distribution plot with normal fit
ax = sns.displot(data = train_raw['SalePrice'], kde=True)
plt.show()

### Test de normalité de Shapiro-Wilk

Pour confirmer objectivement ce que le graphique laisse entrevoir, nous appliquons le **test de Shapiro-Wilk**, un test statistique formel de normalité. Son hypothèse nulle (H₀) est que les données suivent une loi normale.

- Si la **p-value < α = 0.05**, on rejette H₀ : la distribution n'est **pas normale**.
- Si la **p-value ≥ α**, on ne peut pas rejeter H₀.

Ici, la p-value est extrêmement faible (proche de 0), ce qui confirme sans ambiguïté que `SalePrice` ne suit pas une loi normale. Le Q-Q plot normal montre clairement des déviations importantes dans les queues de distribution.

En comparaison, le Q-Q plot log-normal montre un bien meilleur alignement avec la diagonale, suggérant qu'une **transformation logarithmique** du prix de vente est appropriée.

In [ ]:
# Shapiro-Wilk Test for normality together with the Q-Q plot
alpha = 0.05
W, p = stats.shapiro(train_raw['SalePrice'])

print('The assocated p-value is : ' + str(p))

if p < alpha : 
    print('With a threshold α = ' + str(alpha) + ', we reject the null hypothesis')
else : 
    print('With a threshold α = ' + str(alpha) + ', we fail to reject the null hypothesis')
    
qqplot(train_raw['SalePrice'], dist = stats.distributions.norm, fit = True, line = '45')
plt.show()

Comparons maintenant avec une distribution **log-normale** : si les points s'alignent mieux sur la diagonale, cela confirme qu'une transformation logarithmique est appropriée.

In [ ]:
qqplot(train_raw['SalePrice'], dist = stats.distributions.lognorm, fit = True, line = '45')
plt.show()

### Transformation logarithmique de la variable cible

Nous appliquons `np.log1p` (équivalent à log(1 + x)) plutôt que `np.log` pour éviter tout problème si une valeur était égale à 0. Cette transformation logarithmique présente plusieurs avantages :

1. **Symétrise la distribution** : la nouvelle variable `target = log(1 + SalePrice)` suit une distribution beaucoup plus proche d'une gaussienne, ce qui satisfait les hypothèses de la régression linéaire.
2. **Réduit l'influence des valeurs extrêmes** : les biens très chers ont moins d'impact disproportionné sur le modèle.
3. **Interprétation économique** : travailler sur le log du prix revient à modéliser des **variations relatives** (en pourcentage), ce qui est cohérent avec la manière dont les marchés immobiliers fonctionnent en pratique.

> **Note importante** : lors de la soumission finale, il faudra appliquer la transformation inverse `np.expm1` pour retrouver les prix en dollars.

In [ ]:
# new variable target which is the logarithm of the SalePrice
target = np.log1p(train_raw['SalePrice'])

# distribution plot
ax = sns.displot(data = target, kde=True)
plt.show()

### Stratégie d'imputation des valeurs manquantes

#### Variables quantitatives

Pour les variables quantitatives, la documentation indique que les valeurs manquantes signifient **l'absence de la caractéristique** (pas de piscine, pas de véranda...). Par conséquent, remplacer les `NaN` par **0** est la stratégie la plus cohérente : une surface de 0 m² reflète fidèlement l'absence de l'équipement.

Notons que cette approche est préférable à l'imputation par la moyenne ou la médiane, qui introduirait une information fictive et biaiserait les corrélations.

#### Variables qualitatives

Pour les variables catégorielles, le même raisonnement s'applique : les `NaN` ne sont pas des données manquantes au sens statistique, mais l'indication que la caractéristique n'existe pas. On les remplace donc par la modalité `'NA'` (chaîne de caractères), qui devient une catégorie à part entière dans l'encodage.

In [ ]:
def plot_missing(data):

    missing = data.isnull().sum()
    missing = missing[missing > 0]
    
    if missing.empty:
        print('No missing values')
    else :
        missing.sort_values(inplace=True)
        missing.plot.bar()
    
        plt.show()

### Catégorisation des variables

Avant d'analyser les valeurs manquantes et de préparer les données, nous classons les 79 variables prédictives en trois groupes :

- **Variables qualitatives** (53 variables) : variables catégorielles décrivant des caractéristiques non ordonnées (matériaux, style, zonage…). Elles ne peuvent pas être utilisées directement dans une régression et nécessiteront un encodage.
- **Variables quantitatives** (21 variables) : mesures numériques continues (surfaces en pieds carrés, valeurs monétaires…). Ces variables peuvent être utilisées directement, mais il faut vérifier leur échelle.
- **Variables de date** (4 variables) : années de construction, de rénovation, de construction du garage et de vente. Utilisées brutes, ces valeurs sont peu informatives ; leur transformation en **ancienneté** (différence avec l'année de vente) sera plus pertinente.

Cette trichotomie guide l'ensemble de la stratégie de prétraitement.

In [ ]:
qualitative = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
              'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
              'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
              'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
              'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
              'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
              'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageFinish', 
              'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
              'SaleType', 'SaleCondition', 'MoSold']

quantitative = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'MasVnrArea', 'BsmtFinSF1',  
               'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 
               'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 
               'PoolArea', 'MiscVal']

date = ['YearBuilt', 'YearRemodAdd', 'YrSold', 'GarageYrBlt']

### Fusion des jeux d'entraînement et de validation

Nous fusionnons les deux jeux de données avec `pd.concat` pour traiter les valeurs manquantes et effectuer les transformations de manière **cohérente et simultanée**. L'enjeu est crucial :

- Si l'on traitait les données séparément, on risquerait d'obtenir des transformations différentes (e.g., valeurs de remplacement différentes selon le jeu).
- Certaines valeurs manquantes n'apparaissent **que dans le jeu de validation** (données de production) et seraient ignorées si l'on ne traitait que l'entraînement.

> Cette approche reflète un problème réel en production : les pipelines de prétraitement doivent être conçus pour gérer toutes les données futures de manière robuste.

In [ ]:
data_raw = pd.concat([train_raw, validation_raw])

In [ ]:
plot_missing(data_raw[quantitative])

In [ ]:
plot_missing(data_raw[date])

In [ ]:
qualitative = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
              'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
              'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
              'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
              'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
              'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
              'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageFinish', 
              'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
              'SaleType', 'SaleCondition', 'MoSold']

quantitative = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'MasVnrArea', 'BsmtFinSF1',  
               'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 
               'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 
               'PoolArea', 'MiscVal']

date = ['YearBuilt', 'YearRemodAdd', 'YrSold', 'GarageYrBlt']

In [ ]:
data_raw = pd.concat([train_raw, validation_raw])

## 1.3 Transformations de base

Avant de modéliser, certaines variables doivent être réexprimées. Dans un contexte fiscal, ce qui importe n'est pas l'année absolue de construction (1962 ou 2003), mais l'**ancienneté au moment de la vente** : c'est elle qui capte la dépréciation et les besoins de rénovation.

Ces transformations améliorent également la **généralisation** : un modèle entraîné en 2008 qui raisonne en ancienneté reste valide en 2015, contrairement à un modèle sur années absolues.

### Variables de date

Le seul cas particulier parmi les variables de date est `GarageYrBlt`, qui présente des valeurs manquantes pour les biens sans garage. Nous l'imputons avec `YearBuilt` (l'année de construction de la maison), ce qui est logique : si un garage a été ajouté lors de la construction sans date connue, l'année de la maison est la meilleure approximation disponible.

Ensuite, nous transformons ces dates en **ancienneté** : au lieu de l'année absolue (e.g., 1960), nous calculons le nombre d'années écoulées depuis la vente (e.g., 48 ans). Cette transformation est plus pertinente car :
- Elle capture l'effet de dépréciation dans le temps.
- Elle rend la variable indépendante de l'année de référence, ce qui améliore la généralisation du modèle.

In [ ]:
def compute_differences_to_year_sold(data) :
    
    data_clean = data.copy()
    
    data_clean['YearBuilt'] = data_clean['YrSold'] - data_clean['YearBuilt']
    data_clean['YearRemodAdd'] = data_clean['YrSold'] - data_clean['YearRemodAdd']
    data_clean['GarageYrBlt'] = data_clean['YrSold'] - data_clean['GarageYrBlt']
    
    return data_clean

In [ ]:
def fill_missing_with_constant(data, columns, constant):
    data_clean = data.copy()
    data_clean[columns] = data_clean[columns].fillna(constant)
    return data_clean

def fill_missing_with_column(data, missing, column):
    data_clean = data.copy()
    for m, c in zip(missing, column):
        data_clean[m] = data_clean[m].fillna(data_clean[c])
    return data_clean

La fonction `clean` consolide en un seul appel toutes les étapes de nettoyage : imputation des valeurs manquantes, remplacement des `GarageYrBlt` manquants, et conversion des dates en ancienneté.

In [ ]:
def clean(data) :
    
    data_clean = data.copy()
    
    # imputing missing variables
    data_clean = fill_missing_with_constant(data_clean, columns = quantitative, constant = 0)
    data_clean = fill_missing_with_constant(data_clean, columns = qualitative, constant = 'NA')
    data_clean = fill_missing_with_column(data_clean, missing = ['GarageYrBlt'], column = ['YearBuilt'])

    # transform date columns
    data_clean = compute_differences_to_year_sold(data_clean)
    
    return data_clean

Appliquons ce pipeline sur l'ensemble des données (train + validation) pour obtenir un jeu propre et cohérent.

In [ ]:
data_clean = clean(data_raw)

data_clean.head()

# 2. Ingénierie des données et du modèle

Passons à présent à la construction du modèle. Nous débuterons avec un modèle de référence simple avant d'augmenter progressivement la complexité.

## 2.1 Encodage des variables catégorielles : Target Mean Encoding

Plutôt qu'un encodage one-hot (explosion de dimensionnalité), nous utilisons le **target mean encoding** : chaque modalité est remplacée par la **moyenne du log-prix** pour les biens de cette catégorie.

**Risque de data leakage** : l'encodage est calculé sur les données d'entraînement uniquement, ce qui est correct. Utiliser la validation pour calculer ces moyennes introduirait du leakage.

In [ ]:
def encode_with_mean(data, target, features):
    
    data_preprocess = data.copy()
    
    for f in features: 
        
        # create a temporary dataframe for our workload
        frame = pd.DataFrame()
        frame[f] = data[f].copy()
        frame[target.name] = target.copy()
        
        # create the mapping table
        mapping = pd.DataFrame()
        mapping['val'] = data[f].unique()
        mapping.index = mapping.val
        
        # compute the mean of our target variable for each category
        mapping['mean'] = frame[[f, target.name]].groupby(f).mean()[target.name]
        
        # if a category has NA, we shall simply put the mean value
        mapping['mean'] = mapping['mean'].fillna(target.mean())

        # we replace the feature with the means in the mapping table
        data_preprocess[f] = pd.merge(data_preprocess, mapping, left_on = f, right_index = True)['mean'].copy()
        
    return data_preprocess

Nous distinguons maintenant les variables **catégorielles** (à encoder par la moyenne) des **numériques** (à standardiser), puis séparons le jeu d'entraînement du jeu de validation.

In [ ]:
categorical = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
               'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
               'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
               'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
               'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
               'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
               'KitchenQual','TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 
               'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 
               'MiscFeature', 'SaleType', 'SaleCondition', 'MoSold']

numeric = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'MasVnrArea', 'BsmtFinSF1',
           'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 
           'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 
           'PoolArea', 'MiscVal', 'YearBuilt', 'YearRemodAdd', 'YrSold', 'GarageYrBlt']

# Split of the clean dataset into train & validation
train_clean = data_clean[data_clean.index.isin(train_raw.index)]
validation_clean = data_clean[data_clean.index.isin(validation_raw.index)]

### Analyse des corrélations avec la variable cible

Nous calculons le **coefficient de corrélation de Pearson** entre chaque variable numérique et le logarithme du prix de vente. Ce coefficient mesure la force et la direction d'une relation linéaire, entre -1 (corrélation négative parfaite) et +1 (corrélation positive parfaite).

Les variables les plus corrélées positivement avec le log-prix sont typiquement :
- `OverallQual` : la qualité globale du bien — ce qui confirme l'intuition : plus un bien est de qualité, plus il est cher.
- `GrLivArea` : la surface habitable totale.
- `TotalBsmtSF`, `1stFlrSF` : les surfaces des différents niveaux.
- `GarageArea`, `GarageCars` : la taille et la capacité du garage.

Inversement, les corrélations négatives (ancienneté du bien, nombre d'années depuis la rénovation) s'interprètent facilement : un bien plus ancien vaut généralement moins cher.

Ces corrélations guident notre sélection des variables pour le modèle de référence.

In [ ]:
def correlation(y, X, features, method = 'pearson'):
    
    cor = pd.DataFrame()
    cor['feature'] = features
    
    cor['correlation_coef'] = [X[f].corr(y, method = method) for f in features]
    cor['correlation_coef'] = cor['correlation_coef'].fillna(0)
    
    cor = cor.sort_values('correlation_coef', ascending = False)

    plt.figure(figsize=(10, 0.25*len(features)))
    sns.barplot(data = cor, y = 'feature', x = 'correlation_coef', orient = 'h')
    
    return cor

Appliquons cette fonction sur nos données d'entraînement pour obtenir le classement des variables par corrélation avec le log-prix.

In [ ]:
cor = correlation(target, train_clean, numeric)

On constate que de nombreuses variables sont corrélées avec le **logarithme du prix de vente**, ce qui est un signal encourageant pour notre modélisation. Examinons maintenant les corrélations internes entre variables. Nous limitons la visualisation aux variables présentant une corrélation modérée ou plus élevée (coefficient supérieur à 0,5 en valeur absolue).

Les corrélations internes sont nombreuses. Il sera donc nécessaire de sélectionner soigneusement les variables afin d'éviter les problèmes de multicolinéarité dans notre modèle.

### Visualisation des relations individuelles : scatter plots

Les corrélations agrégées masquent la **forme** de la relation. Un nuage de points avec droite de régression permet de détecter :

- **La non-linéarité** : l'effet de la surface peut avoir des rendements décroissants.
- **L'hétéroscédasticité** : la variabilité du prix pourrait croître avec sa valeur.
- **Les outliers** : ventes atypiques (saisies, ventes familiales) qui biaiseraient le modèle.

Dans le contexte fiscal, identifier ces non-linéarités est crucial : un seuil fiscal crée une discontinuité artificielle que le modèle doit naviguer sans erreurs systématiques.

In [ ]:
def scatter_plots(y, X, features) :
    
    for f in features :
        x = X[f]
        
        plt.title('Correlation ' + y.name + ' & ' + x.name)
        sns.regplot(x = x.name, y = y.name, data = pd.concat([x, y], axis = 1), x_jitter = .05)
        
        plt.show()

### Analyse des variables catégorielles : box plots et ANOVA

Pour les variables catégorielles, il n'est pas possible de calculer directement une corrélation. Nous utilisons deux outils :

**Box plots** : pour chaque modalité (ex. chaque quartier `Neighborhood`), on visualise la distribution du log-prix. Un bon prédicteur présentera des boîtes bien séparées.

**ANOVA** : teste si les moyennes de log-prix diffèrent significativement selon les modalités.

**Implication métier** : le quartier est l'archétype de la variable catégorielle déterminante. Deux biens identiques en surface mais dans des quartiers différents peuvent se retrouver de part et d'autre d'un seuil fiscal.

In [ ]:
def box_plots(y, X, features) :
    
    for f in features :
        x = X[f]
    
        plt.title('Box plot ' + y.name + ' & ' + x.name)
        sns.boxplot(x = x, y = y)
        x = plt.xticks(rotation = 90)
        
        plt.show()

In [ ]:
data_encoded = encode_with_mean(data_clean, target, categorical)
train_preprocess = data_encoded[data_encoded.index.isin(train_raw.index)]
validation_preprocess = data_encoded[~data_encoded.index.isin(train_raw.index)]

## 2.2 Modèle de référence (Baseline OLS)

### Sélection des variables du modèle de référence

Nous construisons un premier modèle avec seulement **5 variables**, choisies pour leur forte corrélation avec le prix et leur interprétabilité métier :

| Variable | Justification |
|---|---|
| `OverallQual` | Qualité globale notée de 1 à 10 par un expert |
| `Neighborhood` | Le quartier — facteur clé en immobilier |
| `GrLivArea` | Surface habitable totale au-dessus du sol |
| `TotalBsmtSF` | Surface totale du sous-sol |
| `YearBuilt` | Ancienneté du bien |

Ce modèle de référence servira de **benchmark** : tout modèle plus complexe devra faire mieux pour justifier sa complexité supplémentaire.

In [ ]:
base = ['OverallQual', 'Neighborhood', 'GrLivArea', 'TotalBsmtSF', 'YearBuilt']

X = train_preprocess[base]
X_val = validation_preprocess[base]

y = target.copy()

### Métrique d'évaluation : RMSE (Root Mean Squared Error)

Nous utilisons le **RMSE** calculé sur le **logarithme du prix**, ce qui mesure des **erreurs relatives** (~10 % d'erreur pour un RMSE de 0.1). Cela est plus équitable que le RMSE sur les prix bruts, qui pénaliserait davantage les erreurs sur les biens chers.

In [ ]:
from sklearn.metrics import mean_squared_error

def rmse(actual, predicted) :
    return np.sqrt(mean_squared_error(actual, predicted))

### Interprétation des résultats OLS

**Qualité globale :** R² = 0.825 — le modèle explique 82,5 % de la variance avec 5 variables.

**Coefficients (sur le log-prix) :**
- `OverallQual` (coef = 0.090) : +1 point de qualité ≈ **+9.4 %** de prix.
- `Neighborhood` (coef = 0.349) : un bon quartier peut représenter **+41 %** de valeur.
- `YearBuilt` (coef = -0.0015) : coefficient négatif car la variable représente l'**ancienneté** — plus le bien est ancien, moins il vaut.

Le condition number élevé (1.2e+05) signale une multicolinéarité — attendue car les surfaces des différents niveaux sont naturellement corrélées.

In [ ]:
from statsmodels.graphics.gofplots import ProbPlot
plt.rc('font', size=14)
plt.rc('figure', titlesize=18)
plt.rc('axes', labelsize=15)
plt.rc('axes', titlesize=18)

def graph(formula, x_range, label=None):
    """
    Helper function for plotting cook's distance lines
    """
    x = x_range
    y = formula(x)
    plt.plot(x, y, label=label, lw=1, ls='--', color='red')


def diagnostic_plots(X, y, model_fit=None):
    """
    Function to reproduce the 4 base plots of an OLS model in R.

    ---
    Inputs:

    X: A numpy array or pandas dataframe of the features to use in building the linear regression model
    y: A numpy array or pandas series/dataframe of the target variable of the linear regression model

    model_fit [optional]: a statsmodel.api.OLS model after regressing y on X. If not provided, will be
                        generated from X, y
    """

    if not model_fit:
        model_fit = sm.OLS(y, sm.add_constant(X)).fit()

    # create dataframe from X, y for easier plot handling
    dataframe = pd.concat([X, y], axis=1)
    
    # model values
    model_fitted_y = model_fit.fittedvalues
    # model residuals
    model_residuals = model_fit.resid
    # normalized residuals
    model_norm_residuals = model_fit.get_influence().resid_studentized_internal
    # absolute squared normalized residuals
    model_norm_residuals_abs_sqrt = np.sqrt(np.abs(model_norm_residuals))
    # absolute residuals
    model_abs_resid = np.abs(model_residuals)
    # leverage, from statsmodels internals
    model_leverage = model_fit.get_influence().hat_matrix_diag
    # cook's distance, from statsmodels internals
    model_cooks = model_fit.get_influence().cooks_distance[0]
    
    plot_lm_1 = plt.figure()
    plot_lm_1.axes[0] = sns.residplot(x = model_fitted_y, y = dataframe.columns[-1], data = dataframe,
                                      lowess=True,
                                      scatter_kws={'alpha': 0.5},
                                      line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8})
    
    plot_lm_1.axes[0].set_title('Residuals vs Fitted')
    plot_lm_1.axes[0].set_xlabel('Fitted values')
    plot_lm_1.axes[0].set_ylabel('Residuals');
    
    # annotations
    abs_resid = model_abs_resid.sort_values(ascending=False)
    abs_resid_top_3 = abs_resid[:3]
    for i in abs_resid_top_3.index:
        plot_lm_1.axes[0].annotate(i,
                                   xy=(model_fitted_y[i],
                                       model_residuals[i]));
        
    QQ = ProbPlot(model_norm_residuals)
    plot_lm_2 = QQ.qqplot(line='45', alpha=0.5, color='#4C72B0', lw=1)
    plot_lm_2.axes[0].set_title('Normal Q-Q')
    plot_lm_2.axes[0].set_xlabel('Theoretical Quantiles')
    plot_lm_2.axes[0].set_ylabel('Standardized Residuals');
    
    # annotations
    abs_norm_resid = np.flip(np.argsort(np.abs(model_norm_residuals)), 0)
    abs_norm_resid_top_3 = abs_norm_resid[:3]
    for r, i in enumerate(abs_norm_resid_top_3):
        plot_lm_2.axes[0].annotate(i,
                                   xy=(np.flip(QQ.theoretical_quantiles, 0)[r],
                                       model_norm_residuals[i]));
    
    plot_lm_3 = plt.figure()
    plt.scatter(model_fitted_y, model_norm_residuals_abs_sqrt, alpha=0.5);
    sns.regplot(x = model_fitted_y, y = model_norm_residuals_abs_sqrt,
                scatter=False,
                ci=False,
                lowess=True,
                line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8});
    plot_lm_3.axes[0].set_title('Scale-Location')
    plot_lm_3.axes[0].set_xlabel('Fitted values')
    plot_lm_3.axes[0].set_ylabel(r'$\sqrt{|Standardized Residuals|}$');
    
    # annotations
    abs_sq_norm_resid = np.flip(np.argsort(model_norm_residuals_abs_sqrt), 0)
    abs_sq_norm_resid_top_3 = abs_sq_norm_resid[:3]
    for i in abs_norm_resid_top_3:
        plot_lm_3.axes[0].annotate(i,
                                   xy=(model_fitted_y[i],
                                       model_norm_residuals_abs_sqrt[i]));
    
    plot_lm_4 = plt.figure();
    plt.scatter(model_leverage, model_norm_residuals, alpha=0.5);
    sns.regplot(x = model_leverage, y = model_norm_residuals,
                scatter=False,
                ci=False,
                lowess=True,
                line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8});
    plot_lm_4.axes[0].set_xlim(0, max(model_leverage)+0.01)
    plot_lm_4.axes[0].set_ylim(-3, 5)
    plot_lm_4.axes[0].set_title('Residuals vs Leverage')
    plot_lm_4.axes[0].set_xlabel('Leverage')
    plot_lm_4.axes[0].set_ylabel('Standardized Residuals');
    
    # annotations
    leverage_top_3 = np.flip(np.argsort(model_cooks), 0)[:3]
    for i in leverage_top_3:
        plot_lm_4.axes[0].annotate(i,
                                   xy=(model_leverage[i],
                                       model_norm_residuals[i]));
        
    p = len(model_fit.params) # number of model parameters
    graph(lambda x: np.sqrt((0.5 * p * (1 - x)) / x),
          np.linspace(0.001, max(model_leverage), 50),
          'Cook\'s distance') # 0.5 line
    graph(lambda x: np.sqrt((1 * p * (1 - x)) / x),
          np.linspace(0.001, max(model_leverage), 50)) # 1 line
    plot_lm_4.legend(loc='upper right');

### Validation des hypothèses du modèle OLS : graphiques de diagnostic

La régression OLS repose sur 4 hypothèses. Leur violation signifie que les estimations ne sont plus fiables — ce qui, dans un contexte fiscal, pourrait biaiser certaines évaluations.

| Graphique | Hypothèse testée | Ce qu'on cherche |
|---|---|---|
| **Residuals vs Fitted** | Linéarité & homoscédasticité | Nuage aléatoire autour de 0 |
| **Normal Q-Q** | Normalité des résidus | Points alignés sur la diagonale |
| **Scale-Location** | Variance constante | Ligne rouge horizontale |
| **Residuals vs Leverage** | Influence des observations | Pas de points au-delà de Cook |

La **distance de Cook** identifie les ventes atypiques (saisies, ventes familiales) qui fausseraient les estimations fiscales.

In [ ]:
from statsmodels.graphics.gofplots import ProbPlot
plt.rc('font', size=14)
plt.rc('figure', titlesize=18)
plt.rc('axes', labelsize=15)
plt.rc('axes', titlesize=18)

def graph(formula, x_range, label=None):
    """
    Helper function for plotting cook's distance lines
    """
    x = x_range
    y = formula(x)
    plt.plot(x, y, label=label, lw=1, ls='--', color='red')


def diagnostic_plots(X, y, model_fit=None):
    """
    Function to reproduce the 4 base plots of an OLS model in R.

    ---
    Inputs:

    X: A numpy array or pandas dataframe of the features to use in building the linear regression model
    y: A numpy array or pandas series/dataframe of the target variable of the linear regression model

    model_fit [optional]: a statsmodel.api.OLS model after regressing y on X. If not provided, will be
                        generated from X, y
    """

    if not model_fit:
        model_fit = sm.OLS(y, sm.add_constant(X)).fit()

    # create dataframe from X, y for easier plot handling
    dataframe = pd.concat([X, y], axis=1)
    
    # model values
    model_fitted_y = model_fit.fittedvalues
    # model residuals
    model_residuals = model_fit.resid
    # normalized residuals
    model_norm_residuals = model_fit.get_influence().resid_studentized_internal
    # absolute squared normalized residuals
    model_norm_residuals_abs_sqrt = np.sqrt(np.abs(model_norm_residuals))
    # absolute residuals
    model_abs_resid = np.abs(model_residuals)
    # leverage, from statsmodels internals
    model_leverage = model_fit.get_influence().hat_matrix_diag
    # cook's distance, from statsmodels internals
    model_cooks = model_fit.get_influence().cooks_distance[0]
    
    plot_lm_1 = plt.figure()
    plot_lm_1.axes[0] = sns.residplot(x = model_fitted_y, y = dataframe.columns[-1], data = dataframe,
                                      lowess=True,
                                      scatter_kws={'alpha': 0.5},
                                      line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8})
    
    plot_lm_1.axes[0].set_title('Residuals vs Fitted')
    plot_lm_1.axes[0].set_xlabel('Fitted values')
    plot_lm_1.axes[0].set_ylabel('Residuals');
    
    # annotations
    abs_resid = model_abs_resid.sort_values(ascending=False)
    abs_resid_top_3 = abs_resid[:3]
    for i in abs_resid_top_3.index:
        plot_lm_1.axes[0].annotate(i,
                                   xy=(model_fitted_y[i],
                                       model_residuals[i]));
        
    QQ = ProbPlot(model_norm_residuals)
    plot_lm_2 = QQ.qqplot(line='45', alpha=0.5, color='#4C72B0', lw=1)
    plot_lm_2.axes[0].set_title('Normal Q-Q')
    plot_lm_2.axes[0].set_xlabel('Theoretical Quantiles')
    plot_lm_2.axes[0].set_ylabel('Standardized Residuals');
    
    # annotations
    abs_norm_resid = np.flip(np.argsort(np.abs(model_norm_residuals)), 0)
    abs_norm_resid_top_3 = abs_norm_resid[:3]
    for r, i in enumerate(abs_norm_resid_top_3):
        plot_lm_2.axes[0].annotate(i,
                                   xy=(np.flip(QQ.theoretical_quantiles, 0)[r],
                                       model_norm_residuals[i]));
    
    plot_lm_3 = plt.figure()
    plt.scatter(model_fitted_y, model_norm_residuals_abs_sqrt, alpha=0.5);
    sns.regplot(x = model_fitted_y, y = model_norm_residuals_abs_sqrt,
                scatter=False,
                ci=False,
                lowess=True,
                line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8});
    plot_lm_3.axes[0].set_title('Scale-Location')
    plot_lm_3.axes[0].set_xlabel('Fitted values')
    plot_lm_3.axes[0].set_ylabel('$\sqrt{|Standardized Residuals|}$');
    
    # annotations
    abs_sq_norm_resid = np.flip(np.argsort(model_norm_residuals_abs_sqrt), 0)
    abs_sq_norm_resid_top_3 = abs_sq_norm_resid[:3]
    for i in abs_norm_resid_top_3:
        plot_lm_3.axes[0].annotate(i,
                                   xy=(model_fitted_y[i],
                                       model_norm_residuals_abs_sqrt[i]));
    
    plot_lm_4 = plt.figure();
    plt.scatter(model_leverage, model_norm_residuals, alpha=0.5);
    sns.regplot(x = model_leverage, y = model_norm_residuals,
                scatter=False,
                ci=False,
                lowess=True,
                line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8});
    plot_lm_4.axes[0].set_xlim(0, max(model_leverage)+0.01)
    plot_lm_4.axes[0].set_ylim(-3, 5)
    plot_lm_4.axes[0].set_title('Residuals vs Leverage')
    plot_lm_4.axes[0].set_xlabel('Leverage')
    plot_lm_4.axes[0].set_ylabel('Standardized Residuals');
    
    # annotations
    leverage_top_3 = np.flip(np.argsort(model_cooks), 0)[:3]
    for i in leverage_top_3:
        plot_lm_4.axes[0].annotate(i,
                                   xy=(model_leverage[i],
                                       model_norm_residuals[i]));
        
    p = len(model_fit.params) # number of model parameters
    graph(lambda x: np.sqrt((0.5 * p * (1 - x)) / x),
          np.linspace(0.001, max(model_leverage), 50),
          'Cook\'s distance') # 0.5 line
    graph(lambda x: np.sqrt((1 * p * (1 - x)) / x),
          np.linspace(0.001, max(model_leverage), 50)) # 1 line
    plot_lm_4.legend(loc='upper right');

#### Analyse des résidus

Les résidus semblent exempts de toute structure apparente et paraissent distribués normalement autour de 0. On note toutefois la présence de quelques observations atypiques. Examinons ces données de plus près.

La documentation du jeu de données (http://jse.amstat.org/v19n3/decock/DataDocumentation.txt) indique :

_NOTES PARTICULIÈRES :_
_Il existe 5 observations qu'un instructeur pourrait souhaiter retirer du jeu de données avant de le soumettre aux étudiants (un graphique du PRIX DE VENTE en fonction de GR LIV AREA permettra de les identifier rapidement). Trois d'entre elles sont de véritables valeurs aberrantes (ventes partielles ne reflétant probablement pas la valeur réelle du marché) et deux correspondent à des ventes atypiques (très grandes maisons avec des prix somme toute cohérents). Il est conseillé de retirer les maisons dépassant 4 000 pieds carrés du jeu de données._

Examinons ces valeurs extrêmes, en particulier sur la colonne **GrLivArea**.

Nous allons maintenant réexécuter la même analyse en excluant les valeurs aberrantes identifiées.

In [ ]:
def plot_prediction(y_actual, y_predicted):
    plt.figure(figsize=(8, 6))
    plt.scatter(y_actual, y_predicted, alpha=0.4)
    min_val = min(y_actual.min(), y_predicted.min())
    max_val = max(y_actual.max(), y_predicted.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1)
    plt.xlabel('Valeurs réelles (log)')
    plt.ylabel('Valeurs prédites (log)')
    plt.title('Réel vs Prédit')
    plt.tight_layout()
    plt.show()

y_pred = res.predict(sm.add_constant(X))

Le résultat est nettement meilleur. Évaluons à présent notre modèle de référence afin d'estimer ses performances sur Kaggle. Nous utiliserons la méthode **predict** pour effectuer les prédictions.

Visualisons le nuage de points comparant les valeurs réelles aux valeurs prédites par le modèle.

In [ ]:
plot_prediction(y, y_pred)

## 2.3 Soumission du modèle de référence sur Kaggle

Nous appliquons la méthode **predict** sur le jeu de données de validation.

Nous enregistrons nos prédictions dans le fichier _baseline.csv_. Il ne faut pas oublier d'appliquer la fonction inverse de **np.log1p**, à savoir **np.expm1**.

<img src="Img/baseline.jpg" width=800 />

## 3. Pipeline de production : rechargement propre des données

Nous repartons d'un **état propre** en rechargeant les données brutes depuis les fichiers CSV pour garantir la reproductibilité du pipeline.

> **Analogie production** : dans un système d'évaluation fiscale réel, ce pipeline serait déclenché à chaque mutation immobilière. La reproductibilité est une exigence fonctionnelle.

In [ ]:
train_raw = pd.read_csv('Data/train.csv', index_col = 'Id')
validation_raw = pd.read_csv('Data/test.csv', index_col = 'Id')

# Logarithmic transformation of SalePrice for the training dataset
target = np.log1p(train_raw['SalePrice'])

# We shall drop the orginal column from the training dataset
train_raw = train_raw.drop(['SalePrice'], axis=1)

# We concatenate the two dataset together for a smooth preprocessing
data_raw = pd.concat([train_raw, validation_raw])

Les listes de variables sont identiques à celles de la section 1 — redéfinies ici pour que le pipeline soit auto-suffisant et exécutable indépendamment.

In [ ]:
qualitative = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
              'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
              'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
              'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
              'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
              'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
              'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageFinish', 
              'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
              'SaleType', 'SaleCondition', 'MoSold']

quantitative = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'MasVnrArea', 'BsmtFinSF1',  
               'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 
               'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 
               'PoolArea', 'MiscVal']

date = ['YearBuilt', 'YearRemodAdd', 'YrSold', 'GarageYrBlt']

In [ ]:
def remove_outliers(data, target, feature, threshold=4000):
    mask = data[feature] <= threshold
    return data[mask], target[mask]

La deuxième liste distingue les variables **catégorielles** (à encoder par la moyenne) des **numériques** (à standardiser), utilisées dans le pipeline d'encodage et de mise à l'échelle.

In [ ]:
categorical = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
               'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
               'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
               'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
               'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
               'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
               'KitchenQual','TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 
               'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 
               'MiscFeature', 'SaleType', 'SaleCondition', 'MoSold']

numeric = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'MasVnrArea', 'BsmtFinSF1',
           'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 
           'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 
           'PoolArea', 'MiscVal', 'YearBuilt', 'YearRemodAdd', 'YrSold', 'GarageYrBlt']

## Pipeline de prétraitement complet

La fonction `preprocessing` consolide toutes les étapes en un seul pipeline reproductible :

1. **Imputation** des valeurs manquantes.
2. **Transformation des dates** en ancienneté.
3. **Encodage par la moyenne** des variables catégorielles.
4. **Suppression des outliers** sur `GrLivArea` — uniquement sur le train.
5. **Standardisation** avec `StandardScaler` : essentielle pour les modèles régularisés.

In [ ]:
from sklearn.preprocessing import StandardScaler

def preprocessing(data, target) :
    
    data_preprocess = data.copy()
    
    target_preprocess = target.copy()
    
    # imputing missing variables
    data_preprocess = fill_missing_with_constant(data_preprocess, columns = quantitative, constant = 0)
    data_preprocess = fill_missing_with_constant(data_preprocess, columns = qualitative, constant = 'NA')
    data_preprocess = fill_missing_with_column(data_preprocess, missing = ['GarageYrBlt'], column = ['YearBuilt'])

    # transform date columns
    data_preprocess = compute_differences_to_year_sold(data_preprocess)
    
    # encode with mean
    data_preprocess = encode_with_mean(data_preprocess, target_preprocess, categorical)
    
    # split between train & validation 
    train_preprocess = data_preprocess[data_preprocess.index.isin(target_preprocess.index)]
    validation_preprocess = data_preprocess[~data_preprocess.index.isin(target_preprocess.index)]
    
    # remove outliers from train 
    train_preprocess, target_preprocess = remove_outliers(train_preprocess, target_preprocess, feature = 'GrLivArea')
    
    # concatenate the two dataset back together
    data_preprocess = pd.concat([train_preprocess, validation_preprocess])
    
    # scale using the standard scaler
    scaler = StandardScaler()
    data_scale = scaler.fit_transform(data_preprocess.to_numpy(copy = True))
    
    # transform back to a pandas dataframe
    data_preprocess = pd.DataFrame(data_scale, index = data_preprocess.index, columns = data_preprocess.columns)

    return data_preprocess, target_preprocess

### Données prêtes pour la modélisation

Après prétraitement, les données forment une matrice numérique standardisée avec **79 variables prédictives**, centrées autour de 0.

Le jeu d'entraînement (`X`, `y`) et le jeu de validation (`X_val`) sont maintenant prêts pour tout algorithme : Ridge/Lasso, Random Forest, Gradient Boosting, etc.

In [ ]:
data_preprocess, target_preprocess = preprocessing(data_raw, target)

y = target_preprocess.copy()
X = data_preprocess[data_preprocess.index.isin(target_preprocess.index)]
X_val = data_preprocess[~ data_preprocess.index.isin(target_preprocess.index)]

X.head()

## 4. Prochaines étapes : amélioration du modèle

**Pistes d'amélioration :**
- **Régularisation** (Ridge / Lasso) : réduit la variance face à la multicolinéarité.
- **Modèles ensemblistes** (Random Forest, Gradient Boosting) : capturent les non-linéarités.
- **Ingénierie de variables** : combiner des variables (ex. surface totale) pour révéler des patterns.

**Critère de succès** : améliorer significativement le RMSE de référence (~0.15 sur le log-prix, soit ~15 % d'erreur relative).